In [1]:
# At some point we will make this a utlity function in the qiskit-ibm-runtime package itself
def get_api_key(db_path, table_name, column_name, condition):
    import sqlite3
    """
    Retrieves an API key from an SQLite database.
    :param db_path: Path to the SQLite database file.
    :param table_name: Name of the table containing the API key.
    :param column_name: Column name where the API key is stored.
    :param condition: SQL condition to fetch the correct API key (e.g., "id=1").
    :return: API key as a string or None if not found.
    """
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        query = f"SELECT {column_name} FROM {table_name} WHERE {condition}"
        cursor.execute(query)
        result = cursor.fetchone()
        
        conn.close()
        
        return result[0] if result else None
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return None

In [2]:
# Get IBM Quantum API key 
# we need to simplify this 
db_path = "C:\\Users\\patri\\OneDrive\\Projects\\admin\\api_store.db" 
table = "api_keys"
column_id = "key" 
condition = "service='ibm_quantum'"
api_key = get_api_key(db_path, table, column_id, condition)
if api_key is None:
    raise ValueError("API key not found in the database.")

# Expectation value 

In [6]:
import numpy as np
from qiskit.circuit.library import IQP
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp, random_hermitian
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
 
n_qubits = 120  # Number of qubits for the IQP circuit, adjust as needed
 
service = QiskitRuntimeService(channel="ibm_quantum", token=api_key)
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)
 
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = IQP(mat)
observable = SparsePauliOp("Z" * n_qubits)  # Define the observable as Z^n_qubits for the expectation value
 
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)
 
estimator = Estimator(mode=backend)
job = estimator.run([(isa_circuit, isa_observable)])
job_id = job.job_id()
print(f" > Job ID: {job_id}")

# Wait for the job to complete and get the result
result = job.result()
 
print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")

 > Job ID: czmkh1wpj4bg008z1vc0
 > Expectation value: -0.00732421875
 > Metadata: {'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}
